# Business Scenario

In the fast-paced world of e-commerce, NoorMart, a rapidly scaling platform operating across the MENA region receives over 150,000 customer interactions daily. This feedback is scattered across App Store reviews, Twitter (X) mentions, and direct support tickets. Currently, analyzing this localized, dialect-heavy Arabic feedback is a highly manual, slow, and expensive process. By the time a human agent spots a viral complaint regarding a delayed delivery, the PR damage is already done.

Imagine a scenario where a state-of-the-art AI system automatically reads, cleans, and categorizes every single customer message in real-time. By leveraging advanced Deep Learning (Stacked Bi-LSTMs) and robust Arabic word embeddings (AraVec), this project aims to develop an automated sentiment analysis microservice.

This system can be seamlessly integrated into NoorMart's backend architecture. As soon as a tweet or review is posted, the API instantly classifies the sentiment as Positive, Negative, or Neutral. This AI-powered solution offers significant advantages:

- Immediate Escalation: Instantly routes "Negative" complaints to the VIP Customer Support team to prevent customer churn.

- Automated Marketing: Aggregates "Positive" reviews for promotional campaigns.

- Resource Optimization: Reduces the need for manual ticket tagging, allowing human agents to focus on resolving actual complex issues.

**Evaluation Metric**: In customer service, missing a negative complaint is a catastrophic failure. Because the "Neutral" class naturally dominates everyday interactions, standard Accuracy is a misleading metric (a model could guess "Neutral" every time and score highly). Therefore, this project strictly optimizes for the F1-Score, ensuring the model heavily penalizes False Negatives and maintains high precision and recall across all customer emotions.

## 1. Importing Core Libraries
This cell imports all the necessary packages for the project:
* **pandas & numpy:** For data manipulation and numerical operations.
* **re:** For regular expressions used in text cleaning.
* **tensorflow:** The main deep learning framework.
* **sklearn.model_selection:** For splitting the data into train, validation, and test sets.

In [1]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split

2025-11-13 14:38:14.603989: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-13 14:38:14.779290: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763037494.846293  105299 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763037494.866063  105299 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763037495.010737  105299 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## 2. Loading and Preparing Data

### 2.1. Loading Dataset 1 (Arabic Reviews)
We'll start by loading the first dataset, which consists of 100,000 Arabic reviews in a TSV file.

In [2]:
# 1. Load the dataset
df0 = pd.read_csv("arabicsentdata/ar_reviews_100k.tsv" , sep='\t')
df0.head()

,label,text
0,Positive,ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...
1,Positive,أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...
2,Positive,هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...
3,Positive,خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...
4,Positive,ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...


### 2.2. Standardizing Columns
To ensure consistency when merging datasets, we rename the `label` and `text` columns to `sentiment` and `Text`, respectively.

In [3]:
df0= df0.rename(columns={'label': 'sentiment', 'text': 'Text'})

### 2.3. Inspecting Labels (Dataset 1)
We check the unique values in the `sentiment` column to see what labels are used (e.g., "Positive", "Mixed", "Negative").

In [4]:
df0['sentiment'].unique()

array(['Positive', 'Mixed', 'Negative'], dtype=object)

### 2.4. Mapping Labels (Dataset 1)
We standardize the labels to a common format ("positive", "negative", "neutral") to match the second dataset.

In [5]:
label_map = {
    "Positive": "positive",
    "Negative": "negative",
    "Mixed": "neutral"
}
# Apply the mapping. Use .replace() which is robust.
# We apply it to the 'sentiment' column
df0['sentiment'] = df0['sentiment'].replace(label_map)
df0['sentiment'].unique()

array(['positive', 'neutral', 'negative'], dtype=object)

### 2.5. Loading Dataset 2 (Arabic Tweets)
Next, we load the second dataset, which contains general Arabic tweets.

In [6]:
df1 = pd.read_csv("arabicsentdata/train_all.csv")
df1.head()

,Tweet_id,sentiment,Text
0,1221884257490042887,neutral,الزعل بيغير ملامحك بيغير نظرة العين بيغير شك...
1,1221884400377499651,neutral,@halgawi @DmfMohe ليس حباً في ايران بقدر ماهو ...
2,1221881406168731649,neutral,@adalfahadduwail أبي أعرف الحاكم العربي المسلم...
3,1221882047691657217,neutral,@sarmadbouchamou @DimaSadek في الخطاب تبع سليم...
4,1221880371773673474,neutral,@FofaMahmouddd مفيش الكلام ده في الزمن


### 2.6. Inspecting Labels (Dataset 2)
We verify that the labels in this dataset are already in the standard format we need.

In [7]:
df1['sentiment'].unique()

array(['neutral', 'negative', 'positive'], dtype=object)

### 2.7. Merging Datasets
We combine the two dataframes (`df0` and `df1`) into a single, large dataframe (`df`) that will be split later on to train, validation and test sets.

In [8]:
df = pd.concat([df0, df1], ignore_index=True)

### 2.8. Reviewing the Merged Data
We display the head and shape of the final merged dataset, which now contains 154,999 instances.

In [9]:
df

,sentiment,Text,Tweet_id
0,positive,ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...,NaN
1,positive,أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...,NaN
2,positive,هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...,NaN
3,positive,خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...,NaN
4,positive,ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...,NaN
...,...,...,...
154994,positive,@mhrsd_care السلام عليكم متى سيتم استئناف دوام...,1.254332e+18
154995,positive,@tfrabiah #وزير_الصحه #وزارة_الصحة_السعودية @e...,1.252259e+18
154996,positive,بصراحة أشكر @MOCSaudi على البثوث المميزة والسه...,1.253082e+18
154997,positive,ان شاء الله مثل هذا القرار يطبق عندنا #السعودي...,1.253332e+18


### 2.9. Checking Class Distribution
We check the value counts for the `sentiment` column to understand the class balance. The dataset is slightly imbalanced, with 'neutral' being the largest class.

In [ ]:
df['sentiment'].value_counts()

sentiment
neutral     70692
positive    42154
negative    42153
Name: count, dtype: int64

## 3. Text Preprocessing

### 3.1. Defining the Arabic Cleaning Function
We define a function `clean_arabic` to normalize the text. This function:
1.  Removes links, mentions, hashtags, and non-Arabic characters.
2.  Strips Arabic diacritics (Tashkeel).
4.  Removes punctuation and extra whitespace.
5.  Applies this cleaning function to the entire 'Text' column.

In [ ]:
# Define the set of diacritics to remove
diacritics = re.compile(r'[\u064b-\u065e\u0670\u0610-\u061a\u06d6-\u06ed]', re.UNICODE)

def clean_arabic(text):
    text = str(text)

    # 1. Standard Cleaning (similar to yours)
    text = re.sub(r"http\S+|www.\S+|[@#]\S+|[A-Za-z0-9]", "", text)
    
    # 2. Diacritics Removal (Crucial for lowering vocab size)
    text = diacritics.sub('', text)

    # 3. Orthography Normalization
    # Unify different forms of Alif to simple Alif ('ا')
    text = re.sub(r'[إأآ]', 'ا', text)
    # Unify different forms of Ya and Waw
    text = re.sub(r'[يى]', 'ي', text) 
    text = re.sub(r'[ؤئ]', 'ء', text) 
    # Unify Ta' Marbuta ('ة') to Ha ('ه') OR to Ta' ('ت') (depends on task; here we use 'ت' for better stemming compatibility)
    text = re.sub(r'ة', 'ت', text)
    
    # 4. Punctuation Removal (Only keep Arabic letters and spaces)
    # Note: \u0600-\u06FF is the range for general Arabic script
    text = re.sub(r"[^\u0600-\u06FF\s]", "", text)
    
    # 5. Normalize spaces and strip
    text = re.sub(r"\s+", " ", text).strip()
    
    # Remove single-letter noise tokens (optional, but helpful if they're not meaningful prepositions)
    # text = ' '.join([word for word in text.split() if len(word) > 1])

    return text

df['Text'] = df['Text'].apply(clean_arabic)

### 3.2. Checking for Empty Rows
After cleaning, some text rows might become empty (e.g., if they only contained mentions and links). We check how many such rows exist.

In [ ]:
empty_texts = df[df['Text'].str.strip() == ""]
print(f"Empty rows: {len(empty_texts)}")

Empty rows: 6


### 3.3. Dropping Empty/NA Rows
To prevent errors during training, we remove all rows that have missing values (NaN) or that became empty strings after the cleaning process.

In [ ]:
# Drop missing or empty text rows before splitting
df = df.dropna(subset=['Text', 'sentiment'])
df = df[df['Text'].str.strip().astype(bool)]

### 3.4. Verifying Clean-up
We run the check one more time to confirm that all empty rows have been successfully dropped.

In [ ]:
empty_texts = df[df['Text'].str.strip() == ""]
print(f"Empty rows: {len(empty_texts)}")

Empty rows: 0


### 3.5. Splitting the Data (Train/Val/Test)
We split the data into three sets using a two-step stratified split to maintain the class distribution:
1.  **Step 1:** Split into 90% Training and 10% Temporary.
2.  **Step 2:** Split the 10% Temporary set in half, creating a 5% Validation set and a 5% Test set.

In [ ]:
# 4. Train/validation split
# 1️⃣ Split into train + temp (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['Text'],
    df['sentiment'],
    test_size=0.1,  # 20% = validation + test
    stratify=df['sentiment'],
    shuffle=True,
    random_state=42
)

# 2️⃣ Split temp into val + test (10% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,  # half of 20% = 10%
    stratify=y_temp,
    shuffle=True,
    random_state=42
)


### 3.6. Reviewing Split Sizes
We print the final number of samples in each set to confirm the 90/5/5 split.

In [ ]:
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 139493, Val: 7750, Test: 7750


### 3.7. Analyzing Sequence Lengths
To decide on an appropriate `max_len` for padding, we first calculate the length (in words) of every sentence in the dataset.

In [ ]:
# Length of each sentence in words
seq_lengths = df['Text'].str.split().str.len()

### 3.8. Determining 99th Percentile Length
We find the 99th percentile of sequence lengths. This tells us a length that will cover 99% of all sentences, helping us balance performance and computational cost.

In [ ]:
# Choose max_len to cover 99% of sentences
max_len_95 = int(np.percentile(seq_lengths, 99))
print("Recommended max_len for 99% coverage:", max_len_95)

Recommended max_len for 99% coverage: 351


### 3.9. Calculating Total Unique Vocabulary
We calculate the total number of unique words (vocabulary size) in the entire cleaned dataset.

In [ ]:
# 1. Split every sentence in the 'Text' column into a list of words (tokens).
token_lists = df['Text'].str.split()

# 2. Flatten the list of lists into a single, long list of all words.
#    We use numpy for efficient concatenation/flattening.
all_words = np.concatenate(token_lists.values)

# 3. Use the 'set' function to get only the unique words.
unique_vocab = set(all_words)

# 4. Return the size of the set.
len(unique_vocab)

360529

### 3.10. Calculating 95% Vocabulary Coverage
The total vocabulary is large (360k+). To build a more efficient model, we find the number of unique words needed to cover 95% of all *word occurrences*. This significantly reduces the vocabulary size while keeping the most common words.

In [ ]:
from collections import Counter
# 1. Split all sentences into tokens
token_lists = df['Text'].str.split()
all_tokens = np.concatenate(token_lists.values)

# 2. Count word frequencies
token_counts = Counter(all_tokens)

# 3. Sort by frequency
sorted_tokens = token_counts.most_common()

# 4. Compute cumulative coverage
total_tokens = sum(token_counts.values())
cum_counts = np.cumsum([count for _, count in sorted_tokens])
coverage = cum_counts / total_tokens

# 5. Find vocab size that covers 95% of all tokens
target_coverage = 0.95
vocab_size_95 = np.searchsorted(coverage, target_coverage) + 1

print("Recommended vocab size for 95% coverage:", vocab_size_95)


Recommended vocab size for 95% coverage: 107884


### 3.11. Initializing and Adapting the TextVectorization Layer
We define our Keras `TextVectorization` layer.
* We set `max_tokens` to the 95% coverage size (107,884).
* We set `output_sequence_length` to 512, which will pad or truncate all sequences to this length.(however we precalculated max_len for 99% coverage to be 351 , we can increase it to 512 for some long sentences) 
* We then `adapt` (fit) the vectorizer to the training text to build its vocabulary.

In [ ]:
# 5. Define TextVectorization layer
vocab_size = 107884
max_len = 512

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=max_len,
    standardize=None,  # important for Arabic — don't strip characters
    split='whitespace',
    
)

# Adapt vectorizer to training text
vectorizer.adapt(X_train)

vocab = vectorizer.get_vocabulary()
word_index = dict(zip(vocab, range(len(vocab))))

I0000 00:00:1763037551.085060  105299 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5561 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


### 3.12. Initializing and Adapting the StringLookup Layer
We create a `StringLookup` layer to convert our string labels ("positive", "negative", "neutral") into integer indices (0, 1, 2) that the model can understand.

In [ ]:
label_lookup = tf.keras.layers.StringLookup(num_oov_indices=0)

label_lookup.adapt(y_train)

## 4. Creating the Data Pipeline

### 4.1. Creating tf.data.Dataset Objects
We convert our pandas splits (X_train, y_train, etc.) into `tf.data.Dataset` objects, which are highly efficient for training in TensorFlow.

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

### 4.2. Defining the Preprocessing Function
We create a function that will be applied to our `tf.data` pipeline. It uses the already-adapted `vectorizer` and `label_lookup` layers to convert text and labels into their final numerical, one-hot encoded format.

In [ ]:
def preprocessing_fn(dataset):
    
    dataset_vectorized = dataset.map(
        lambda text, label: (
            vectorizer(text), 
            tf.one_hot(label_lookup(label), depth=3)
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return dataset_vectorized

### 4.3. Applying the Preprocessing Function
We use the `.map()` method to apply our `preprocessing_fn` to all three datasets.

In [ ]:
# Preprocess the train and test data
train_dataset_vectorized = preprocessing_fn(train_dataset)
val_dataset_vectorized = preprocessing_fn(val_dataset)
test_dataset_vectorized = preprocessing_fn(test_dataset)

### 4.4. Inspecting the Pipeline Output
We take 3 samples from the training pipeline to verify that the text has been converted to padded integer vectors and the labels are correctly one-hot encoded.

In [ ]:
for text_vector, label in train_dataset_vectorized.take(3):
    print("Text (vectorized):", text_vector.numpy())
    print("Label (encoded):", label.numpy())
    print("---")

Text (vectorized): [19279   541   356     5  3062 22115  8252   890     4   116   541  8327
  6746  8252 20103     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0  

2025-11-13 14:39:14.509532: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### 4.5. Optimizing the Data Pipelines
We apply `.cache()`, `.shuffle()`(for training set only), `.prefetch()`, and `.batch()` to our datasets. This is a critical step for performance, as it allows the GPU to train without waiting for the CPU to load and preprocess new batches.

In [ ]:
SHUFFLE_BUFFER_SIZE = 1000
PREFETCH_BUFFER_SIZE = tf.data.AUTOTUNE
BATCH_SIZE = 64

# Optimize and batch the datasets for training
train_dataset_final = (train_dataset_vectorized
                       .cache()
                       .shuffle(SHUFFLE_BUFFER_SIZE)
                       .prefetch(PREFETCH_BUFFER_SIZE)
                       .batch(BATCH_SIZE)
                       )

val_dataset_final = (val_dataset_vectorized
                      .cache()
                      .prefetch(PREFETCH_BUFFER_SIZE)
                      .batch(BATCH_SIZE)
                      )

test_dataset_final = (test_dataset_vectorized
                      .cache()
                      .prefetch(PREFETCH_BUFFER_SIZE)
                      .batch(BATCH_SIZE)
                      )

## 5. Building the Model

### 5.1. Loading Pre-trained AraVec Embeddings
We load the pre-trained AraVec (Word2Vec) model. This provides our model with a strong understanding of Arabic word relationships from the start.

In [ ]:
from gensim.models import Word2Vec

# Load AraVec pretrained Twitter CBOW model
embedding_model = Word2Vec.load("pre_embed/full_grams_cbow_300_twitter.mdl")

# Now build a compatible embedding_index dictionary
embedding_index = {word: embedding_model.wv[word] for word in embedding_model.wv.key_to_index}

### 5.2. Creating the Custom Embedding Matrix
We create a weight matrix that maps the vocabulary from our `TextVectorization` layer (word_index) to the corresponding vectors from the AraVec model.

In [ ]:
embedding_dim = 300  # because AraVec / FastText is 300-d

embedding_matrix = np.zeros((len(vocab), embedding_dim))

for word, i in word_index.items():
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

### 5.3. Defining the Stacked Bi-LSTM Model
This is the core of our project. We build a `Sequential` model:
1.  **Input:** A fixed-length input shape.
2.  **Embedding Layer:** Uses our custom `embedding_matrix` and is set to `trainable=True` to allow fine-tuning.
3.  **SpatialDropout1D:** Regularization to prevent overfitting on the embedding layer.
4.  **Stacked Bi-LSTMs:** Two `Bidirectional LSTM` layers. The first returns sequences, feeding into the second, which returns a single summary vector.
5.  **Classification Head:** `Dense` layers with `Dropout` to classify the LSTM's output.
6.  **Compiler:** We compile the model with an Adam optimizer and `f1_score` as our metric.

In [ ]:
# 6. Build the model
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(max_len,)),
    tf.keras.layers.Embedding(input_dim=vocab_size,
                              output_dim=embedding_dim,
                              weights=[embedding_matrix], 
                              trainable=True,
                              embeddings_regularizer=None),
    tf.keras.layers.SpatialDropout1D(0.3),    
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64 , return_sequences=True)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')                   # assuming 3 sentiment classes
])

optim = tf.keras.optimizers.Adam(learning_rate=0.0001, weight_decay=0.005)
model.compile(loss='categorical_crossentropy', optimizer=optim, metrics=['f1_score'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 512, 300)       │    32,365,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 512, 300)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 512, 128)       │       186,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,659,347 (124.59 MB)

 Trainable params: 32,659,347 (124.59 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Training and Evaluation

### 6.1. Training the Model
We define our callbacks:
* **EarlyStopping:** Stops training if the `val_loss` doesn't improve for 5 epochs and restores the best weights.
* **ReduceLROnPlateau:** Reduces the learning rate if the `val_loss` plateaus.
We then launch the training process using `model.fit()`.

In [ ]:
# 8. Train the model

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                               patience=5, 
                                               mode="min",
                                               restore_best_weights=True)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,          # Reduce learning rate by 50%
    patience=2,          # If F1 hasn't improved in 2 epochs, reduce LR
    mode='min',
    min_lr=1e-6          # Don't let the LR go below 1e-6
)
                                         
history = model.fit(train_dataset_final, validation_data=val_dataset_final, epochs=50, callbacks=[early_stop , lr_scheduler ])

Epoch 1/50


I0000 00:00:1763037578.841928  105885 cuda_dnn.cc:529] Loaded cuDNN version 91200


2180/2180 ━━━━━━━━━━━━━━━━━━━━ 325s 147ms/step - f1_score: 0.4678 - loss: 0.9456 - val_f1_score: 0.6521 - val_loss: 0.7392 - learning_rate: 1.0000e-04
Epoch 2/50
2180/2180 ━━━━━━━━━━━━━━━━━━━━ 320s 147ms/step - f1_score: 0.6406 - loss: 0.7657 - val_f1_score: 0.6761 - val_loss: 0.7109 - learning_rate: 1.0000e-04
Epoch 3/50
2180/2180 ━━━━━━━━━━━━━━━━━━━━ 330s 152ms/step - f1_score: 0.6675 - loss: 0.7232 - val_f1_score: 0.6826 - val_loss: 0.6853 - learning_rate: 1.0000e-04
Epoch 4/50
2180/2180 ━━━━━━━━━━━━━━━━━━━━ 322s 148ms/step - f1_score: 0.6810 - loss: 0.6973 - val_f1_score: 0.6943 - val_loss: 0.6728 - learning_rate: 1.0000e-04
Epoch 5/50
2180/2180 ━━━━━━━━━━━━━━━━━━━━ 316s 145ms/step - f1_score: 0.6944 - loss: 0.6753 - val_f1_score: 0.6994 - val_loss: 0.6655 - learning_rate: 1.0000e-04
Epoch 6/50
2180/2180 ━━━━━━━━━━━━━━━━━━━━ 327s 150ms/step - f1_score: 0.7050 - loss: 0.6588 - val_f1_score: 0.7110 - val_loss: 0.6556 - learning_rate: 1.0000e-04
Epoch 7/50
2180/2180 ━━━━━━━━━━━━━━━━━━

### 6.2. Final Evaluation on Test Set
After training, the model (with the best weights restored) is evaluated on the unseen test set. This gives us our final, unbiased performance metrics.

In [ ]:
# Model Evaluation 

print("\n--- Running Final Model Evaluation ---")
results = model.evaluate(test_dataset_final, verbose=1)

final_f1 = results[1].numpy().flatten()[0]
print("\n--- Final Test Set Evaluation ---")
print(f"Test Loss: {results[0]:.4f}")
print(f"Test F1 Score: {final_f1:.4f}")


--- Running Final Model Evaluation ---
122/122 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step - f1_score: 0.6988 - loss: 0.6632

--- Final Test Set Evaluation ---
Test Loss: 0.6531
Test F1 Score: 0.7307


### 6.3. Saving the Final Model
We save the complete model (architecture and weights) to a single `.keras` file for later use in deployment.

In [ ]:
model.save('arabic_sentiment_model.keras')

In [ ]:
# Example input
sentence = "المنتج سيء جدا و غير مطابق للمواصفات"

# Apply your robust Arabic cleaning
sentence_clean = clean_arabic(sentence)

sentence_vectorized = vectorizer(np.array([sentence_clean]))

pred_probs = model.predict(sentence_vectorized)

label_vocab = ['neutral', 'positive', 'negative']

pred_probs = model.predict(sentence_vectorized)
pred_class_idx = np.argmax(pred_probs[0])
pred_class = label_vocab[pred_class_idx]
pred_confidence = pred_probs[0][pred_class_idx]

print(f"Predicted sentiment: {pred_class} ({pred_confidence:.2f})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
Predicted sentiment: negative (0.95)
